In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import MACCSkeys, Descriptors,AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize
from matplotlib.colors import ListedColormap
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
import rdkit.Chem.Descriptors3D
from morfeus import xtb, read_xyz
import os

In [5]:
file_path = './MixtureSolDB.csv'

solubility_df = pd.read_csv(file_path)

In [7]:
solubility_df.head()

,SMILES_Solute,Temperature_K,Solubility(mole_fraction),LogS(mole_fraction),Solubility(g/100g),LogS(g/100g),Solvent1,Solvent2,SMILES_Solvent1,SMILES_Solvent2,Fraction_Solvent1,Fraction_Solvent2,Fraction_Type,Compound_Name,CAS,PubChem_CID,FDA_Approved,Source,IsPureSolventEndpoint
0,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,278.15,0.0872,-1.059484,212.339615,2.327031,acetonitrile,water,CC#N,O,0.0,1.0,mole,Rivastigmine tartrate,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True
1,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,283.15,0.0983,-1.007446,242.315737,2.384382,acetonitrile,water,CC#N,O,0.0,1.0,mole,Rivastigmine tartrate,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True
2,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,288.15,0.1126,-0.948462,282.038981,2.450309,acetonitrile,water,CC#N,O,0.0,1.0,mole,Rivastigmine tartrate,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True
3,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,293.15,0.1287,-0.890421,328.322771,2.516301,acetonitrile,water,CC#N,O,0.0,1.0,mole,Rivastigmine tartrate,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True
4,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,298.15,0.1469,-0.832978,382.747203,2.582912,acetonitrile,water,CC#N,O,0.0,1.0,mole,Rivastigmine tartrate,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True


In [8]:
# Первичный аудит структуры данных и пропусков

print("Shape:", solubility_df.shape)
print("\nColumns:")
print(solubility_df.columns.tolist())

print("\nDtypes:")
print(solubility_df.dtypes)

print("\nMissing values:")
missing_df = pd.DataFrame({
    'missing_count': solubility_df.isna().sum(),
    'missing_percent': (solubility_df.isna().mean() * 100).round(2)
}).sort_values('missing_count', ascending=False)
print(missing_df)

print("\nNumber of duplicated rows:", solubility_df.duplicated().sum())

print("\nFraction_Type value counts:")
print(solubility_df['Fraction_Type'].value_counts(dropna=False))

print("\nIsPureSolventEndpoint value counts:")
print(solubility_df['IsPureSolventEndpoint'].value_counts(dropna=False))

key_cols = [
    'SMILES_Solute', 'Temperature_K', 'Solubility(mole_fraction)',
    'Solvent1', 'Solvent2', 'SMILES_Solvent1', 'SMILES_Solvent2',
    'Fraction_Solvent1', 'Fraction_Solvent2', 'Fraction_Type',
    'Compound_Name', 'Source', 'IsPureSolventEndpoint'
]

print("\nMissing values in key columns:")
key_missing_df = pd.DataFrame({
    'missing_count': solubility_df[key_cols].isna().sum(),
    'missing_percent': (solubility_df[key_cols].isna().mean() * 100).round(2)
}).sort_values('missing_count', ascending=False)
print(key_missing_df)

print("\nUnique counts in selected columns:")
for col in ['Solvent1', 'Solvent2', 'Fraction_Type', 'Source', 'Compound_Name']:
    print(f"{col}: {solubility_df[col].nunique(dropna=True)}")

print("\nSample rows:")
display(solubility_df.head())

Shape: (175166, 19)

Columns:
['SMILES_Solute', 'Temperature_K', 'Solubility(mole_fraction)', 'LogS(mole_fraction)', 'Solubility(g/100g)', 'LogS(g/100g)', 'Solvent1', 'Solvent2', 'SMILES_Solvent1', 'SMILES_Solvent2', 'Fraction_Solvent1', 'Fraction_Solvent2', 'Fraction_Type', 'Compound_Name', 'CAS', 'PubChem_CID', 'FDA_Approved', 'Source', 'IsPureSolventEndpoint']

Dtypes:
SMILES_Solute                 object
Temperature_K                float64
Solubility(mole_fraction)    float64
LogS(mole_fraction)          float64
Solubility(g/100g)           float64
LogS(g/100g)                 float64
Solvent1                      object
Solvent2                      object
SMILES_Solvent1               object
SMILES_Solvent2               object
Fraction_Solvent1            float64
Fraction_Solvent2            float64
Fraction_Type                 object
Compound_Name                 object
CAS                           object
PubChem_CID                  float64
FDA_Approved                    b

,SMILES_Solute,Temperature_K,Solubility(mole_fraction),LogS(mole_fraction),Solubility(g/100g),LogS(g/100g),Solvent1,Solvent2,SMILES_Solvent1,SMILES_Solvent2,Fraction_Solvent1,Fraction_Solvent2,Fraction_Type,Compound_Name,CAS,PubChem_CID,FDA_Approved,Source,IsPureSolventEndpoint
0,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,278.15,0.0872,-1.059484,212.339615,2.327031,acetonitrile,water,CC#N,O,0.0,1.0,mole,Rivastigmine tartrate,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True
1,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,283.15,0.0983,-1.007446,242.315737,2.384382,acetonitrile,water,CC#N,O,0.0,1.0,mole,Rivastigmine tartrate,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True
2,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,288.15,0.1126,-0.948462,282.038981,2.450309,acetonitrile,water,CC#N,O,0.0,1.0,mole,Rivastigmine tartrate,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True
3,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,293.15,0.1287,-0.890421,328.322771,2.516301,acetonitrile,water,CC#N,O,0.0,1.0,mole,Rivastigmine tartrate,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True
4,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,298.15,0.1469,-0.832978,382.747203,2.582912,acetonitrile,water,CC#N,O,0.0,1.0,mole,Rivastigmine tartrate,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True


In [9]:
# Проверка суммы долей
solubility_df['fraction_sum'] = (
    solubility_df['Fraction_Solvent1'] + solubility_df['Fraction_Solvent2']
)

print("Fraction sum stats:")
print(solubility_df['fraction_sum'].describe())

print("\nRows where sum != 1:")
invalid_sum = solubility_df[np.abs(solubility_df['fraction_sum'] - 1) > 1e-6]
print(invalid_sum.shape)

# Проверка диапазона
out_of_bounds = solubility_df[
    (solubility_df['Fraction_Solvent1'] < 0) |
    (solubility_df['Fraction_Solvent1'] > 1) |
    (solubility_df['Fraction_Solvent2'] < 0) |
    (solubility_df['Fraction_Solvent2'] > 1)
]

print("\nOut of bounds fractions:", out_of_bounds.shape)

# Проверка чистых растворителей
pure_check = solubility_df[
    solubility_df['IsPureSolventEndpoint'] &
    ~(
        (solubility_df['Fraction_Solvent1'] == 1) |
        (solubility_df['Fraction_Solvent1'] == 0)
    )
]

print("\nInconsistent pure solvent flags:", pure_check.shape)

Fraction sum stats:
count    1.751660e+05
mean     1.000000e+00
std      3.140611e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: fraction_sum, dtype: float64

Rows where sum != 1:
(0, 20)

Out of bounds fractions: (0, 20)

Inconsistent pure solvent flags: (0, 20)


In [15]:
# список проблемных растворителей
bad_solvents = ['PEG-200', 'PEG-300', 'PEG-400', 'PEG-600', 'PEGDME 250']

# удаляем строки, где они встречаются
solubility_df = solubility_df[
    ~solubility_df['Solvent1'].isin(bad_solvents) &
    ~solubility_df['Solvent2'].isin(bad_solvents)
].reset_index(drop=True)

print("New shape:", solubility_df.shape)

New shape: (173716, 24)


In [16]:
from rdkit import RDLogger
from rdkit.Chem import Descriptors

#RDLogger.DisableLog('rdApp.*') чтобы не сыпались сообщения RDKit

def calc_mw(smiles):
    if pd.isna(smiles):
        return np.nan
    smiles = str(smiles).strip()
    if smiles in ['', '-', 'nan', 'None']:
        return np.nan

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.nan
    return Descriptors.MolWt(mol)

solubility_df['MW_1'] = solubility_df['SMILES_Solvent1'].apply(calc_mw)
solubility_df['MW_2'] = solubility_df['SMILES_Solvent2'].apply(calc_mw)

solubility_df['x1'] = solubility_df['Fraction_Solvent1'].copy()

mask_mass = (
    (solubility_df['Fraction_Type'] == 'mass') &
    solubility_df['MW_1'].notna() &
    solubility_df['MW_2'].notna()
)

w1 = solubility_df.loc[mask_mass, 'Fraction_Solvent1']
mw1 = solubility_df.loc[mask_mass, 'MW_1']
mw2 = solubility_df.loc[mask_mass, 'MW_2']

solubility_df.loc[mask_mass, 'x1'] = (
    (w1 / mw1) / ((w1 / mw1) + ((1 - w1) / mw2))
)

solubility_df['x2'] = 1 - solubility_df['x1']

In [17]:
print("Invalid MW_1:", solubility_df['MW_1'].isna().sum())
print("Invalid MW_2:", solubility_df['MW_2'].isna().sum())

bad_smiles_1 = solubility_df.loc[solubility_df['MW_1'].isna(), 'SMILES_Solvent1'].value_counts().head(20)
bad_smiles_2 = solubility_df.loc[solubility_df['MW_2'].isna(), 'SMILES_Solvent2'].value_counts().head(20)

print("\nBad SMILES_Solvent1:")
print(bad_smiles_1)

print("\nBad SMILES_Solvent2:")
print(bad_smiles_2)

Invalid MW_1: 0
Invalid MW_2: 0

Bad SMILES_Solvent1:
Series([], Name: count, dtype: int64)

Bad SMILES_Solvent2:
Series([], Name: count, dtype: int64)


In [22]:
solubility_df.head(5)

,SMILES_Solute,Temperature_K,Solubility(mole_fraction),LogS(mole_fraction),Solubility(g/100g),LogS(g/100g),Solvent1,Solvent2,SMILES_Solvent1,SMILES_Solvent2,...,CAS,PubChem_CID,FDA_Approved,Source,IsPureSolventEndpoint,fraction_sum,MW_1,MW_2,x1,x2
0,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,278.15,0.0872,-1.059484,212.339615,2.327031,acetonitrile,water,CC#N,O,...,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True,1.0,41.053,18.015,0.0,1.0
1,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,283.15,0.0983,-1.007446,242.315737,2.384382,acetonitrile,water,CC#N,O,...,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True,1.0,41.053,18.015,0.0,1.0
2,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,288.15,0.1126,-0.948462,282.038981,2.450309,acetonitrile,water,CC#N,O,...,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True,1.0,41.053,18.015,0.0,1.0
3,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,293.15,0.1287,-0.890421,328.322771,2.516301,acetonitrile,water,CC#N,O,...,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True,1.0,41.053,18.015,0.0,1.0
4,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,298.15,0.1469,-0.832978,382.747203,2.582912,acetonitrile,water,CC#N,O,...,129101-54-8,6918078.0,True,10.1007/s10973-018-7684-y,True,1.0,41.053,18.015,0.0,1.0


In [24]:
cols_to_drop = [
    'Solubility(g/100g)',
    'LogS(g/100g)',
    'Fraction_Solvent1',
    'Fraction_Solvent2',
    'Fraction_Type',
    'CAS',
    'PubChem_CID',
    'FDA_Approved',
    'Source',
    'fraction_sum'
]

cols_keep = [
    'Compound_Name',
    'SMILES_Solute',
    'Temperature_K',
    'Solubility(mole_fraction)',
    'LogS(mole_fraction)',
    'Solvent1',
    'Solvent2',
    'SMILES_Solvent1',
    'SMILES_Solvent2',
    'x1',
    'x2',
    'IsPureSolventEndpoint'
]

In [25]:
solubility_df = solubility_df[cols_keep].copy()

print("Columns after cleanup:")
print(solubility_df.columns.tolist())
print("Shape:", solubility_df.shape)

Columns after cleanup:
['Compound_Name', 'SMILES_Solute', 'Temperature_K', 'Solubility(mole_fraction)', 'LogS(mole_fraction)', 'Solvent1', 'Solvent2', 'SMILES_Solvent1', 'SMILES_Solvent2', 'x1', 'x2', 'IsPureSolventEndpoint']
Shape: (173716, 12)


In [26]:
# берём только чистые растворители
pure_df = solubility_df[solubility_df['IsPureSolventEndpoint']].copy()

# определяем, какой это растворитель
pure_df['PureSolvent'] = np.where(
    pure_df['x1'] == 1,
    pure_df['Solvent1'],
    pure_df['Solvent2']
)

# SMILES тоже полезно сохранить
pure_df['PureSolvent_SMILES'] = np.where(
    pure_df['x1'] == 1,
    pure_df['SMILES_Solvent1'],
    pure_df['SMILES_Solvent2']
)

# оставляем нужные поля
pure_lookup = pure_df[[
    'SMILES_Solute',
    'PureSolvent',
    'PureSolvent_SMILES',
    'Temperature_K',
    'Solubility(mole_fraction)',
    'LogS(mole_fraction)'
]].copy()

In [27]:
pure_lookup = (
    pure_lookup
    .groupby(['SMILES_Solute', 'PureSolvent', 'Temperature_K'], as_index=False)
    .agg({
        'Solubility(mole_fraction)': 'mean',
        'LogS(mole_fraction)': 'mean'
    })
)

In [28]:
print("Pure lookup shape:", pure_lookup.shape)
print("Unique solutes:", pure_lookup['SMILES_Solute'].nunique())
print("Unique solvents:", pure_lookup['PureSolvent'].nunique())

Pure lookup shape: (19399, 5)
Unique solutes: 688
Unique solvents: 118


In [29]:
pure_lookup.head()

,SMILES_Solute,PureSolvent,Temperature_K,Solubility(mole_fraction),LogS(mole_fraction)
0,Br.Cc1ccc(Sc2ccccc2N2CCNCC2)c(C)c1,2-butanone,283.15,0.000062,-4.207608
1,Br.Cc1ccc(Sc2ccccc2N2CCNCC2)c(C)c1,2-butanone,288.15,0.000084,-4.075721
2,Br.Cc1ccc(Sc2ccccc2N2CCNCC2)c(C)c1,2-butanone,293.15,0.000136,-3.866461
3,Br.Cc1ccc(Sc2ccccc2N2CCNCC2)c(C)c1,2-butanone,298.15,0.000163,-3.787812
4,Br.Cc1ccc(Sc2ccccc2N2CCNCC2)c(C)c1,2-butanone,303.15,0.000210,-3.677781


In [31]:
# pure_lookup: SMILES_Solute, PureSolvent, Temperature_K, Solubility(mole_fraction), LogS(mole_fraction)

pure_lookup = pure_lookup.copy()
pure_lookup = pure_lookup.sort_values(['SMILES_Solute', 'PureSolvent', 'Temperature_K']).reset_index(drop=True)

# словарь: (solute, solvent) -> таблица чистого растворителя
pure_dict = {
    (solute, solvent): grp[['Temperature_K', 'Solubility(mole_fraction)', 'LogS(mole_fraction)']]
    .sort_values('Temperature_K')
    .reset_index(drop=True)
    for (solute, solvent), grp in pure_lookup.groupby(['SMILES_Solute', 'PureSolvent'])
}

def find_nearest_pure(solute_smiles, solvent, temp, tol=1.0):
    key = (solute_smiles, solvent)
    if key not in pure_dict:
        return np.nan, np.nan, np.nan
    
    grp = pure_dict[key]
    temps = grp['Temperature_K'].values
    idx = np.argmin(np.abs(temps - temp))
    diff = abs(temps[idx] - temp)
    
    if diff <= tol:
        row = grp.iloc[idx]
        return row['Solubility(mole_fraction)'], row['LogS(mole_fraction)'], diff
    else:
        return np.nan, np.nan, np.nan

work_df = solubility_df.copy()

# для Solvent1
res1 = work_df.apply(
    lambda row: find_nearest_pure(row['SMILES_Solute'], row['Solvent1'], row['Temperature_K'], tol=1.0),
    axis=1
)
work_df[['S1_pure_solubility', 'S1_pure_logS', 'S1_temp_diff']] = pd.DataFrame(res1.tolist(), index=work_df.index)

# для Solvent2
res2 = work_df.apply(
    lambda row: find_nearest_pure(row['SMILES_Solute'], row['Solvent2'], row['Temperature_K'], tol=1.0),
    axis=1
)
work_df[['S2_pure_solubility', 'S2_pure_logS', 'S2_temp_diff']] = pd.DataFrame(res2.tolist(), index=work_df.index)

# статистика
n_total = len(work_df)
n_missing_s1 = work_df['S1_pure_solubility'].isna().sum()
n_missing_s2 = work_df['S2_pure_solubility'].isna().sum()
n_missing_any = work_df[['S1_pure_solubility', 'S2_pure_solubility']].isna().any(axis=1).sum()
n_missing_both = work_df[['S1_pure_solubility', 'S2_pure_solubility']].isna().all(axis=1).sum()

print(f"Total rows: {n_total}")
print(f"Missing pure solubility for Solvent1: {n_missing_s1} ({n_missing_s1 / n_total:.2%})")
print(f"Missing pure solubility for Solvent2: {n_missing_s2} ({n_missing_s2 / n_total:.2%})")
print(f"Missing at least one pure solubility: {n_missing_any} ({n_missing_any / n_total:.2%})")
print(f"Missing both pure solubilities: {n_missing_both} ({n_missing_both / n_total:.2%})")

Total rows: 173716
Missing pure solubility for Solvent1: 40545 (23.34%)
Missing pure solubility for Solvent2: 42731 (24.60%)
Missing at least one pure solubility: 56414 (32.47%)
Missing both pure solubilities: 26862 (15.46%)


In [32]:
def attach_with_tol(df, pure_dict, tol):
    def find_nearest_pure(solute_smiles, solvent, temp):
        key = (solute_smiles, solvent)
        if key not in pure_dict:
            return np.nan

        grp = pure_dict[key]
        temps = grp['Temperature_K'].values
        idx = np.argmin(np.abs(temps - temp))
        diff = abs(temps[idx] - temp)

        if diff <= tol:
            return grp.iloc[idx]['Solubility(mole_fraction)']
        return np.nan

    tmp = df.copy()

    tmp['S1_pure'] = tmp.apply(
        lambda row: find_nearest_pure(row['SMILES_Solute'], row['Solvent1'], row['Temperature_K']),
        axis=1
    )
    tmp['S2_pure'] = tmp.apply(
        lambda row: find_nearest_pure(row['SMILES_Solute'], row['Solvent2'], row['Temperature_K']),
        axis=1
    )

    n_total = len(tmp)
    n_missing_any = tmp[['S1_pure', 'S2_pure']].isna().any(axis=1).sum()
    n_missing_both = tmp[['S1_pure', 'S2_pure']].isna().all(axis=1).sum()

    return {
        'tol': tol,
        'missing_any': n_missing_any,
        'missing_any_pct': round(n_missing_any / n_total * 100, 2),
        'missing_both': n_missing_both,
        'missing_both_pct': round(n_missing_both / n_total * 100, 2),
    }

results = []
for tol in [1.0, 2.5, 5.0, 10.0]:
    results.append(attach_with_tol(solubility_df, pure_dict, tol))

pd.DataFrame(results)

,tol,missing_any,missing_any_pct,missing_both,missing_both_pct
0,1.0,56414,32.47,26862,15.46
1,2.5,53866,31.01,25302,14.57
2,5.0,52599,30.28,24664,14.20
3,10.0,51950,29.91,24372,14.03


In [33]:
# строки, где не найдена растворимость в чистых (оба NaN)
missing_both_df = work_df[
    work_df[['S1_pure_solubility', 'S2_pure_solubility']].isna().all(axis=1)
]

print("Total missing both:", missing_both_df.shape[0])

# выводим 5 примеров
cols_to_show = [
    'SMILES_Solute',
    'Temperature_K',
    'Solvent1', 'Solvent2',
    'x1', 'x2'
]

display(missing_both_df[cols_to_show].head(5))

Total missing both: 26862


,SMILES_Solute,Temperature_K,Solvent1,Solvent2,x1,x2
3521,CC(=O)Oc1cccc2c1C(=O)c1c(OC(C)=O)cc(C(=O)O)cc1...,278.15,DMAc,water,0.022459,0.977541
3522,CC(=O)Oc1cccc2c1C(=O)c1c(OC(C)=O)cc(C(=O)O)cc1...,283.15,DMAc,water,0.022459,0.977541
3523,CC(=O)Oc1cccc2c1C(=O)c1c(OC(C)=O)cc(C(=O)O)cc1...,288.15,DMAc,water,0.022459,0.977541
3524,CC(=O)Oc1cccc2c1C(=O)c1c(OC(C)=O)cc(C(=O)O)cc1...,293.15,DMAc,water,0.022459,0.977541
3525,CC(=O)Oc1cccc2c1C(=O)c1c(OC(C)=O)cc(C(=O)O)cc1...,298.15,DMAc,water,0.022459,0.977541


In [34]:
def check_presence(row):
    key1 = (row['SMILES_Solute'], row['Solvent1'])
    key2 = (row['SMILES_Solute'], row['Solvent2'])
    return (
        key1 in pure_dict,
        key2 in pure_dict
    )

presence = missing_both_df.head(5).apply(check_presence, axis=1)
presence_df = pd.DataFrame(presence.tolist(), columns=['S1_exists', 'S2_exists'])

display(pd.concat([missing_both_df[cols_to_show].head(5).reset_index(drop=True), presence_df], axis=1))

,SMILES_Solute,Temperature_K,Solvent1,Solvent2,x1,x2,S1_exists,S2_exists
0,CC(=O)Oc1cccc2c1C(=O)c1c(OC(C)=O)cc(C(=O)O)cc1...,278.15,DMAc,water,0.022459,0.977541,False,False
1,CC(=O)Oc1cccc2c1C(=O)c1c(OC(C)=O)cc(C(=O)O)cc1...,283.15,DMAc,water,0.022459,0.977541,False,False
2,CC(=O)Oc1cccc2c1C(=O)c1c(OC(C)=O)cc(C(=O)O)cc1...,288.15,DMAc,water,0.022459,0.977541,False,False
3,CC(=O)Oc1cccc2c1C(=O)c1c(OC(C)=O)cc(C(=O)O)cc1...,293.15,DMAc,water,0.022459,0.977541,False,False
4,CC(=O)Oc1cccc2c1C(=O)c1c(OC(C)=O)cc(C(=O)O)cc1...,298.15,DMAc,water,0.022459,0.977541,False,False


In [35]:
work_df.head(5)

,Compound_Name,SMILES_Solute,Temperature_K,Solubility(mole_fraction),LogS(mole_fraction),Solvent1,Solvent2,SMILES_Solvent1,SMILES_Solvent2,x1,x2,IsPureSolventEndpoint,S1_pure_solubility,S1_pure_logS,S1_temp_diff,S2_pure_solubility,S2_pure_logS,S2_temp_diff
0,Rivastigmine tartrate,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,278.15,0.0872,-1.059484,acetonitrile,water,CC#N,O,0.0,1.0,True,0.0207,-1.684030,0.0,0.0872,-1.059484,0.0
1,Rivastigmine tartrate,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,283.15,0.0983,-1.007446,acetonitrile,water,CC#N,O,0.0,1.0,True,0.0241,-1.617983,0.0,0.0983,-1.007446,0.0
2,Rivastigmine tartrate,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,288.15,0.1126,-0.948462,acetonitrile,water,CC#N,O,0.0,1.0,True,0.0280,-1.552842,0.0,0.1126,-0.948462,0.0
3,Rivastigmine tartrate,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,293.15,0.1287,-0.890421,acetonitrile,water,CC#N,O,0.0,1.0,True,0.0321,-1.493495,0.0,0.1287,-0.890421,0.0
4,Rivastigmine tartrate,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,298.15,0.1469,-0.832978,acetonitrile,water,CC#N,O,0.0,1.0,True,0.0372,-1.429457,0.0,0.1469,-0.832978,0.0


In [37]:
# 1) вариант: оставляем только строки, где найдены обе растворимости в чистых
df_drop_missing_pure = work_df[
    work_df['S1_pure_solubility'].notna() &
    work_df['S2_pure_solubility'].notna()
].copy()

# 2) вариант: сохраняем все строки, NaN остаются как есть
df_keep_missing_pure = work_df.copy()

print("Original shape:", work_df.shape)
print("Drop-missing shape:", df_drop_missing_pure.shape)
print("Keep-missing shape:", df_keep_missing_pure.shape)

# сохраняем
df_drop_missing_pure.to_csv("MixtureSolDB_with_pure_only_complete.csv", index=False)
df_keep_missing_pure.to_csv("MixtureSolDB_with_pure_keep_nan.csv", index=False)

# при желании ещё в parquet
# df_drop_missing_pure.to_parquet("MixtureSolDB_with_pure_only_complete.parquet", index=False)
# df_keep_missing_pure.to_parquet("MixtureSolDB_with_pure_keep_nan.parquet", index=False)

Original shape: (173716, 18)
Drop-missing shape: (117302, 18)
Keep-missing shape: (173716, 18)
